# Radar Hipotecario — EDA e Ingeniería de Características
**Proyecto final Diplomado Ciencia de Datos G33 — UNAM FES Acatlán**

Estructura alineada al formato de referencia del curso (Imports → Global variables → Functions → Data Ingestion → Feature Engineering → Visualización).

**Nota de reproducibilidad:** este notebook NO requiere credenciales ni tokens de API. Todos los datos se leen de snapshots públicos versionados en GitHub (`data/snapshots/latest/`), generados por el pipeline de ingesta del repositorio. Esto garantiza que corra igual en cualquier entorno de Google Colab, sin depender de whitelists de IP ni límites de tasa de las APIs originales (Banxico, INEGI).

### Imports

In [ ]:
# Para producción
import io
import json
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

### Global variables

In [ ]:
# Repositorio público — sin tokens, sin credenciales
GITHUB_USER = 'ContrerasPeninsula'
GITHUB_REPO = 'radar-hipotecario'
GITHUB_BRANCH = 'main'

RAW_BASE = f'https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}'

URL_SERIES_BANXICO = f'{RAW_BASE}/data/snapshots/latest/series_banxico.parquet'
URL_UMA = f'{RAW_BASE}/config/uma.json'
URL_REGLAS_INFONAVIT = f'{RAW_BASE}/config/reglas_infonavit_v2026.json'

### Functions

In [ ]:
def cargar_parquet_github(url: str) -> pd.DataFrame:
    """
    Descarga un archivo Parquet desde una URL pública de GitHub (raw.githubusercontent.com)
    y lo carga como DataFrame, sin depender de fsspec/credenciales.
    """
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return pd.read_parquet(io.BytesIO(r.content))

In [ ]:
def cargar_json_github(url: str) -> dict:
    """Descarga y parsea un archivo JSON público de GitHub."""
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

In [ ]:
def calcular_variacion_pct(df: pd.DataFrame, columna: str, periodos: int) -> pd.Series:
    """
    Calcula variación porcentual de una columna a `periodos` observaciones de distancia.
    Para series diarias: periodos=252 aprox. 1 año hábil; periodos=21 aprox. 1 mes.
    """
    return df[columna].pct_change(periods=periodos) * 100

In [ ]:
def resumen_estadistico(df: pd.DataFrame, columnas: list) -> pd.DataFrame:
    """Resumen descriptivo (media, std, min, max, percentiles) de columnas numéricas."""
    return df[columnas].describe().T

## Data Ingestion

### Series macroeconómicas (Banxico: TIIE, Tasa objetivo, FIX, INPC)
Fuente: [Banxico SIE API](https://www.banxico.org.mx/SieAPIRest/) — ingesta programada vía GitHub Actions, snapshot versionado en el repositorio.

In [ ]:
df_banxico = cargar_parquet_github(URL_SERIES_BANXICO)
print(f'Filas: {len(df_banxico):,} | Series: {df_banxico["serie"].nunique()} | '
      f'Rango: {df_banxico["fecha"].min().date()} a {df_banxico["fecha"].max().date()}')
df_banxico.head()

In [ ]:
df_banxico['serie'].value_counts()

In [ ]:
# Verificación de completitud: observaciones por serie y huecos de fechas
df_banxico.groupby('serie')['fecha'].agg(['min', 'max', 'count'])

### UMA (Unidad de Medida y Actualización)
Constante versionada por vigencia — no proviene de una API, se actualiza manualmente cada febrero cuando INEGI/DOF publican el nuevo valor.

In [ ]:
uma_data = cargar_json_github(URL_UMA)
df_uma = pd.DataFrame(uma_data['valores'])
df_uma

### Tabla de tasas Infonavit (motor de reglas, referencia)
No es una serie de tiempo — es la tabla oficial de tasas diferenciadas por nivel salarial (vigente para UMA 2026), usada como insumo del motor de reglas determinista. Se incluye aquí para el EDA porque describe la distribución de tasas que enfrenta cada segmento de usuarios.

In [ ]:
reglas_infonavit = cargar_json_github(URL_REGLAS_INFONAVIT)
df_infonavit_tasas = pd.DataFrame(reglas_infonavit['tasas']['tabla_diferenciada_por_uma'])
print(f"Estado de las reglas: {reglas_infonavit['estado']} | Versión: {reglas_infonavit['version']}")
df_infonavit_tasas.head(10)

## Feature Engineering

### Series macro: pivote a formato ancho y variables derivadas

In [ ]:
# Pivote: una columna por serie, indexado por fecha — facilita features cruzados entre series
df_wide = df_banxico.pivot_table(index='fecha', columns='serie', values='valor').sort_index()
df_wide = df_wide.ffill()  # las series no cotizan todos los mismos días; forward-fill conservador
df_wide.tail()

In [ ]:
# Inflación interanual a partir del INPC (Índice Nacional de Precios al Consumidor)
if 'inpc_general' in df_wide.columns:
    df_wide['inflacion_anual_pct'] = calcular_variacion_pct(df_wide, 'inpc_general', periodos=252)
    df_wide['inflacion_mensual_pct'] = calcular_variacion_pct(df_wide, 'inpc_general', periodos=21)
df_wide[['inpc_general', 'inflacion_mensual_pct', 'inflacion_anual_pct']].tail()

In [ ]:
# Medias móviles de la TIIE — suavizan el ruido de ajustes discretos de política monetaria
df_wide['tiie_28_ma30'] = df_wide['tiie_28'].rolling(window=30).mean()
df_wide['tiie_28_ma90'] = df_wide['tiie_28'].rolling(window=90).mean()
df_wide[['tiie_28', 'tiie_28_ma30', 'tiie_28_ma90']].tail()

In [ ]:
# Spread TIIE vs Tasa objetivo — brecha relevante para detectar presión de mercado
df_wide['spread_tiie_objetivo'] = df_wide['tiie_28'] - df_wide['tasa_objetivo']
df_wide['spread_tiie_objetivo'].describe()

In [ ]:
# Features de fecha, útiles para segmentar el EDA por periodo
df_wide['anio'] = df_wide.index.year
df_wide['mes'] = df_wide.index.month
df_wide['trimestre'] = df_wide.index.quarter
df_wide[['anio', 'mes', 'trimestre']].tail()

### Motor de reglas Infonavit: variables descriptivas de la tabla de tasas

In [ ]:
# Pendiente de la curva de tasas: cuánto sube la tasa por cada UMA adicional de salario
df_infonavit_tasas['delta_tasa'] = df_infonavit_tasas['tasa'].diff()
df_infonavit_tasas[['uma_min', 'uma_max', 'tasa', 'delta_tasa']].describe()

## Visualización

### Tablas

In [ ]:
resumen_estadistico(df_wide, ['tiie_28', 'tasa_objetivo', 'fix_usd', 'inpc_general'])

In [ ]:
# Tabla resumen: rango de tasas Infonavit por decil de UMA
df_infonavit_tasas[['uma_min', 'uma_max', 'salario_mensual_referencia', 'tasa']].iloc[::4]

### Gráficas

In [ ]:
fig = go.Figure()
for serie, nombre in [('tiie_28', 'TIIE 28 días'), ('tasa_objetivo', 'Tasa objetivo Banxico')]:
    fig.add_trace(go.Scatter(x=df_wide.index, y=df_wide[serie], name=nombre, mode='lines'))
fig.update_layout(
    title='Tasas de referencia de Banxico — histórico',
    xaxis_title='Fecha', yaxis_title='Tasa (%)', hovermode='x unified',
)
fig.show()

In [ ]:
fig = px.line(df_wide.reset_index(), x='fecha', y='fix_usd',
              title='Tipo de cambio FIX (pesos por dólar)')
fig.update_layout(xaxis_title='Fecha', yaxis_title='MXN/USD')
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_wide.index, y=df_wide['inflacion_anual_pct'],
                          name='Inflación interanual (%)', mode='lines'))
fig.add_hline(y=3, line_dash='dot', annotation_text='Meta Banxico (3%)', line_color='green')
fig.update_layout(
    title='Inflación interanual (derivada del INPC vía Banxico, serie SP1)',
    xaxis_title='Fecha', yaxis_title='Variación % anual',
)
fig.show()

In [ ]:
fig = px.bar(df_infonavit_tasas, x='salario_mensual_referencia', y='tasa',
             title='Tasa Infonavit diferenciada por nivel salarial (vigente UMA 2026)',
             labels={'salario_mensual_referencia': 'Salario mensual (MXN)', 'tasa': 'Tasa anual'})
fig.update_layout(yaxis_tickformat='.1%')
fig.show()

## Conclusiones preliminares del EDA

- *(completar tras ejecutar en Colab con datos reales — plantilla de hallazgos a validar):*
- La TIIE y la tasa objetivo se mueven en escalones discretos (decisiones de política monetaria), no en tendencia continua — hallazgo relevante para la elección de modelo en la capa predictiva (ver `docs/` y `src/modelos/forecast_tasas.py`).
- La inflación interanual derivada del INPC permite contrastar contra la meta de Banxico (3%) como referencia de contexto macro para el semáforo de decisión.
- La tabla de tasas Infonavit muestra una curva creciente y aproximadamente lineal entre 2.6 y 6.6 UMA de salario — el `delta_tasa` calculado ayuda a cuantificar la progresividad del esquema.